# House Prices · XGBoost tuned + Lasso log diện tích

Thí nghiệm lấy bộ tham số XGBoost tốt nhất trong notebook đính kèm `house-price.ipynb` (cell RandomizedSearchCV). Điểm 0,12872 trong notebook nguồn là trung bình RMSE từ CV dùng để chọn tham số. Ở đây đánh giá lại bằng RMSE OOF trên hai bộ fold của dự án.

### 1. Dữ liệu và cấu hình

Dùng pipeline của dự án. Cố định Lasso log diện tích và tỷ lệ 80/20 để đo tác động khi thay bộ tham số XGBoost. Dùng `ln(SalePrice)` để so sánh trực tiếp với thí nghiệm trước.

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.pipeline import make_pipeline

cwd = Path.cwd().resolve()
ROOT = next(path for path in (cwd, *cwd.parents)
            if (path / "final_model/config.json").is_file())
PREVIOUS = ROOT / "experiments/2026-11-15_Huy_ensemble_log_area"
sys.path.insert(0, str(ROOT))

from final_model.run_model import feature_sets
from src.model import make_lasso, make_xgboost
from src.prepare_data import DATA_DIR, ID, TARGET, make_preprocessor

CONFIG = json.loads((ROOT / "final_model/config.json").read_text(encoding="utf-8"))
SPLITS = {
    "original": "15-11-2026_ensemble_xgboost_lasso_phuc/outputs/20261115_111145_484418",
    "seed123": "15-11-2026_ensemble_seed123_phuc/outputs/20261115_121000_620119",
}
AREA_COLUMNS = ["GrLivArea", "TotalSF"]



train = pd.read_csv(DATA_DIR / "raw/train.csv")
target = np.log(train[TARGET])
output = ROOT / "experiments/2026-11-15_Huy_xgb_tuned_log_area/outputs"
output.mkdir(exist_ok=True)
print(f"Train: {len(train)} dòng")
from xgboost import XGBRegressor

TUNED = {
    "subsample": 0.8, "reg_lambda": 5, "reg_alpha": 0.1,
    "n_estimators": 1500, "min_child_weight": 1, "max_depth": 3,
    "learning_rate": 0.05, "colsample_bytree": 0.8,
}

Train: 1460 dòng


### 2. Đánh giá trên hai bộ fold

Với mỗi fold, lọc ngoại lệ ở phần huấn luyện và fit tiền xử lý trong pipeline. Dùng lại dự đoán OOF của Lasso và ensemble cũ theo `Id`. Đo ba phương án trên cùng 1.460 căn.

In [2]:
def make_inputs(frame):
    """Apply a deterministic transform shared by training and inference."""
    features = feature_sets(frame, CONFIG)
    log_area = features["lasso"].copy()
    log_area[AREA_COLUMNS] = np.log1p(log_area[AREA_COLUMNS])
    return {**features, "log_area_lasso": log_area}


def training_mask(frame):
    rule = CONFIG["outlier_rule"]
    return ~(frame["GrLivArea"].gt(rule["GrLivArea_gt"])
             & frame[TARGET].lt(rule["SalePrice_lt"]))


def fit_predict(name, features, target, fit_rows, prediction_features):
    estimator = (
        make_xgboost(**CONFIG["xgboost"]["params"])
        if name == "xgboost" else make_lasso(**CONFIG["lasso"]["params"])
    )
    model = make_pipeline(make_preprocessor(features), estimator)
    model.fit(features.loc[fit_rows], target.loc[fit_rows])
    return model.predict(prediction_features).astype(float)


def blend(xgb, lasso):
    weights = CONFIG["weights"]
    return weights["xgboost"] * xgb + weights["lasso"] * lasso



features = make_inputs(train)
eligible = training_mask(train)

def fit_tuned(inputs, fit_rows, to_predict):
    pipeline = make_pipeline(
        make_preprocessor(inputs),
        XGBRegressor(**TUNED, tree_method="hist", objective="reg:squarederror",
                     random_state=42, n_jobs=2),
    )
    pipeline.fit(inputs.loc[fit_rows], target.loc[fit_rows])
    return pipeline.predict(to_predict).astype(float)


def rmse(predicted, actual):
    return float(np.sqrt(np.mean((predicted - actual) ** 2)))


results = []
xgb_inputs = features["xgboost"]
for split in SPLITS:
    reference = pd.read_csv(PREVIOUS / "outputs" / f"oof_{split}.csv").set_index(ID)
    reference = reference.loc[train[ID]].reset_index()
    if not np.allclose(reference["log_price"], target, rtol=0, atol=1e-12):
        raise ValueError("Reference predictions do not match the training data.")
    predictions = reference[[ID, "fold", "log_price"]].copy()
    predictions["previous_ensemble"] = reference["log_area"]
    for fold in sorted(predictions["fold"].unique()):
        validation = predictions["fold"].eq(fold)
        predictions.loc[validation, "tuned_xgb"] = fit_tuned(
            xgb_inputs, ~validation & eligible, xgb_inputs.loc[validation]
        )
    predictions["tuned_ensemble"] = blend(predictions["tuned_xgb"], reference["log_area_lasso"])
    predictions.to_csv(output / f"oof_{split}.csv", index=False)
    for scope, rows in [("all", predictions), *predictions.groupby("fold")]:
        summary = {"split": split, "scope": scope, "n": len(rows)}
        for name in ("previous_ensemble", "tuned_xgb", "tuned_ensemble"):
            summary[name] = rmse(rows[name], rows["log_price"])
        results.append(summary)

metrics = pd.DataFrame(results)
metrics.to_csv(output / "metrics.csv", index=False)
print(metrics.loc[metrics["scope"].eq("all")].to_string(index=False, float_format="{:.6f}".format))

   split scope    n  previous_ensemble  tuned_xgb  tuned_ensemble
original   all 1460           0.128776   0.127116        0.125679
 seed123   all 1460           0.126913   0.125850        0.124101


### 3. Nhận xét

Ensemble mới đạt **0,12568** ở bộ fold gốc và **0,12410** ở seed 123, giảm lần lượt 0,00310 và 0,00281 so với ensemble trước. Cả 5 fold của mỗi bộ chia đều cải thiện. XGBoost tuned đứng riêng cũng có RMSE OOF thấp hơn ensemble cũ trên cả hai bộ chia.

Cấu hình XGBoost đến từ quá trình tuning trên cùng bộ dữ liệu Ames. Hai bộ chia đều dùng lại 1.460 căn, nên kết quả cho biết hiệu quả trong CV của dự án. Điểm Kaggle chưa được đo.

### 4. Tạo submission

Fit lại hai nhánh trên tập train sau lọc, trộn dự đoán log, rồi dùng `exp` để đưa về đơn vị giá gốc.

In [3]:
test = pd.read_csv(DATA_DIR / "raw/test.csv")
test_features = make_inputs(test)
xgb_prediction = fit_tuned(features["xgboost"], eligible, test_features["xgboost"])
lasso_prediction = fit_predict(
    "log_area_lasso", features["log_area_lasso"], target, eligible,
    test_features["log_area_lasso"],
)
prices = np.exp(blend(xgb_prediction, lasso_prediction))
if not np.isfinite(prices).all() or not (prices > 0).all():
    raise ValueError("Submission prices must be finite and positive.")

submission = pd.DataFrame({ID: test[ID], TARGET: prices})
sample = pd.read_csv(DATA_DIR / "raw/sample_submission.csv")
if not submission[ID].equals(sample[ID]):
    raise ValueError("Submission IDs do not match the template.")
submission.to_csv(output / "submission.csv", index=False)
print(f"Submission: {len(submission)} rows")


Submission: 1459 rows
